# 01. Модуль 1 — отделение голоса от фона

6 прогонов: 2 модели (U-Net на спектрограмме, Conv-TasNet на waveform) × 3 функции потерь (L1 по спектру, SI-SDR, multi-resolution STFT).
Каждый прогон ограничен 150 минутами (`max_minutes`) и сохраняется после каждой эпохи. Если Colab оборвал сессию —
просто запустите ту же ячейку ещё раз: обучение продолжится с последней эпохи.

In [ ]:
# Подключаем Google Drive, скачиваем код и ставим зависимости (~2 минуты)
from google.colab import drive
drive.mount('/content/drive')

REPO = 'https://github.com/tanyalebedeva/call-audio-analysis.git'
!test -d /content/call-audio-analysis || git clone -q $REPO /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
!mkdir -p $DATA $RUNS
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU не подключён: Среда выполнения → Сменить среду → T4 GPU'

## Обучение
Можно запускать по одной ячейке в день. Начните с `unet_sisdr` — это минимальная рабочая версия.

In [ ]:
!python -m callaudio.train --config configs/separation/unet_sisdr.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/unet_l1.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/unet_mrstft.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/tasnet_sisdr.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/tasnet_l1.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/tasnet_mrstft.yaml

## Оценка на тесте
SI-SDRi, PESQ и STOI по корзинам SNR. `--wer` дополнительно считает WER Whisper на чистой, зашумлённой и очищенной речи — это дольше, поэтому для начала `--limit 300`.

In [ ]:
import glob
for ckpt in sorted(glob.glob(f'{RUNS}/sep_*/best.pt')):
    !python -m callaudio.evaluate separation --ckpt {ckpt} --limit 300 --no-perceptual

In [ ]:
# Полная оценка с PESQ, STOI и WER — для двух лучших моделей
BEST = ['sep_unet_sisdr', 'sep_tasnet_sisdr']
for run in BEST:
    !python -m callaudio.evaluate separation --ckpt {RUNS}/{run}/best.pt --limit 300 --wer --whisper small

## Таблица результатов для записки

In [ ]:
!python -m callaudio.report --runs $RUNS --out $RUNS/../reports
from IPython.display import Markdown
Markdown(open(f'{RUNS}/../reports/separation.md', encoding='utf-8').read())

## Кривые обучения и примеры

In [ ]:
import json, matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for run in sorted(glob.glob(f'{RUNS}/sep_*')):
    rows = [json.loads(l) for l in open(f'{run}/metrics.jsonl')]
    ax.plot([r['epoch'] for r in rows], [r['val_si_sdri'] for r in rows], label=run.split('/')[-1])
ax.set_xlabel('эпоха'); ax.set_ylabel('SI-SDRi на валидации, дБ'); ax.legend(); ax.grid(alpha=.3)
plt.savefig(f'{RUNS}/../reports/separation_curves.png', dpi=150); plt.show()

In [ ]:
# Послушать примеры: смесь, очищенная речь, эталон
from IPython.display import Audio, display
for f in sorted(glob.glob(f'{RUNS}/sep_unet_sisdr/examples/*.wav'))[:6]:
    print(f.split('/')[-1]); display(Audio(f))

In [ ]:
# Фильтры обученного энкодера Conv-TasNet: похожи ли они на синусоиды разных частот (как у STFT)?
import torch
from callaudio.tasks import load_trained
model, _, _ = load_trained(f'{RUNS}/sep_tasnet_sisdr/best.pt')
w = model.encoder.weight.detach()[:, 0]                       # (N, kernel)
order = torch.fft.rfft(w, n=256).abs().argmax(-1).argsort()     # сортируем по доминирующей частоте
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(w[order], aspect='auto', cmap='RdBu'); axes[0].set_title('Фильтры энкодера (по частоте)')
axes[1].imshow(torch.fft.rfft(w[order], n=256).abs(), aspect='auto', origin='lower', cmap='magma'); axes[1].set_title('Их амплитудные спектры')
plt.savefig(f'{RUNS}/../reports/tasnet_filters.png', dpi=150); plt.show()